# Aula 4 · Do Neurônio à Rede

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlexChequer/notebooks-insperai/blob/main/trainees/aula-04-do-neuronio-a-rede.ipynb)

**Trilha de Trainees — InsperAI** · o par prático da [Aula 4](https://trilhas-insperai.vercel.app/trainees/aulas/aula-04/)

---

Você vai montar, treinar e mexer numa rede que reconhece dígitos escritos à mão, os mesmos do vídeo.

**Como funciona:** as células de preparação já estão prontas e você só roda. Onde estiver `TODO`, é com você. Cada tarefa tem uma dica logo acima.

**Regra do dia:** não precisa entender o `perda.backward()` ainda. Isso é backprop, e é a próxima aula.

## 0. Preparação

Os dígitos do scikit-learn são imagens 8x8 (64 pixels), a versão pequena do MNIST que aparece no vídeo.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

torch.manual_seed(0)
np.random.seed(0)

digitos = load_digits()
X, y = digitos.data, digitos.target
print("imagens:", X.shape, "  rótulos:", y.shape, "  valores de 0 a", X.max())

fig, axs = plt.subplots(2, 8, figsize=(10, 3))
for ax, img, rot in zip(axs.ravel(), digitos.images, digitos.target):
    ax.imshow(img, cmap="gray_r")
    ax.set_title(int(rot))
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)

# a mesma normalização que o vídeo usa: pixel entre 0 e 1
X_tr = torch.tensor(X_tr / 16.0, dtype=torch.float32)
X_te = torch.tensor(X_te / 16.0, dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.long)
y_te = torch.tensor(y_te, dtype=torch.long)

print(X_tr.shape, X_te.shape)

Funções de treino. Você não precisa ler linha por linha agora: o miolo é `perda.backward()` mais `otimizador.step()`, que é a aula que vem.

In [ ]:
def treinar(modelo, epocas=60, lr=0.05, lote=64, semente=0, verbose=False, reiniciar=True):
    """Treino simples. O backprop acontece no loss.backward(): é a aula que vem."""
    torch.manual_seed(semente)
    # Sortear pesos novos antes de começar é o que faz rodar a mesma célula duas
    # vezes dar o mesmo número, em vez de continuar de onde o treino anterior
    # parou. O Desafio 2 desliga isso: lá a inicialização é o objeto de estudo,
    # e o sorteio apagaria justamente os zeros que o exercício pede para pôr.
    if reiniciar:
        for camada in modelo:
            if isinstance(camada, nn.Linear):
                camada.reset_parameters()
    otimizador = torch.optim.SGD(modelo.parameters(), lr=lr, momentum=0.9)
    criterio = nn.CrossEntropyLoss()
    historico = {"perda": [], "acuracia": []}
    for epoca in range(epocas):
        modelo.train()
        ordem = torch.randperm(len(X_tr))
        perda_total = 0.0
        for i in range(0, len(ordem), lote):
            idx = ordem[i:i + lote]
            scores = modelo(X_tr[idx])
            perda = criterio(scores, y_tr[idx])
            otimizador.zero_grad()
            perda.backward()
            otimizador.step()
            perda_total += perda.item() * len(idx)
        historico["perda"].append(perda_total / len(ordem))
        historico["acuracia"].append(acuracia(modelo))
        if verbose and (epoca + 1) % 20 == 0:
            print(f"época {epoca+1:3d}  perda {historico['perda'][-1]:.3f}  acurácia {historico['acuracia'][-1]:.3f}")
    return historico


def acuracia(modelo, X=None, y=None):
    X = X_te if X is None else X
    y = y_te if y is None else y
    modelo.eval()
    with torch.no_grad():
        previsto = modelo(X).argmax(dim=1)
    return (previsto == y).float().mean().item()


def n_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

## Tarefa 1 · Monte a rede

A rede recebe 64 pixels e solta 10 scores, um por dígito. No meio vai uma camada escondida com ativação.

Dica: `nn.Sequential(nn.Linear(entradas, escondidos), nn.ReLU(), nn.Linear(escondidos, saidas))`.

In [ ]:
minha_rede = nn.Sequential(
    # TODO: uma camada Linear de 64 para o número de neurônios escondidos que você quiser
    # TODO: uma ativação
    # TODO: uma camada Linear que termine em 10
)

print(minha_rede)
print("parâmetros:", n_parametros(minha_rede))

## Tarefa 2 · Treine e meça

Use `treinar(minha_rede, epocas=100, lr=0.1)` e depois `acuracia(minha_rede)`.

Compare com a regressão logística da célula abaixo. Se a sua rede ficar muito abaixo de 0,95, alguma coisa está estranha.

In [ ]:
from sklearn.linear_model import LogisticRegression

logistica = LogisticRegression(max_iter=5000).fit(X_tr.numpy(), y_tr.numpy())
print(f"regressão logística: {logistica.score(X_te.numpy(), y_te.numpy()):.3f}")

# TODO: treine a sua rede e imprima a acurácia

## Tarefa 3 · Troque a ativação

Monte a mesma rede com `nn.Sigmoid()` e com `nn.Tanh()` e compare as três.

Anote a acurácia e também quantas épocas cada uma levou pra perda cair. A tabela de resposta vai na célula de texto depois.

In [ ]:
resultados = {}
for nome, ativacao in [("ReLU", nn.ReLU()), ("sigmoide", nn.Sigmoid()), ("tanh", nn.Tanh())]:
    modelo = nn.Sequential(nn.Linear(64, 64), ativacao, nn.Linear(64, 10))
    historico = treinar(modelo, epocas=100, lr=0.1)
    resultados[nome] = historico
    print(f"{nome:9s} acurácia {acuracia(modelo):.3f}")

# TODO: plote as três curvas de perda no mesmo gráfico
# dica: plt.plot(historico["perda"], label=nome) dentro do loop, depois plt.legend() e plt.show()

**Sua resposta:** qual ativação chegou mais rápido? A acurácia final mudou muito?

*(escreva aqui)*

## Tarefa 4 · Tire a ativação

Monte a mesma rede sem nenhuma ativação no meio e treine.

In [ ]:
sem_ativacao = nn.Sequential(
    # TODO: duas camadas Linear, sem nada entre elas
)

# TODO: treine e imprima a acurácia

**Sua resposta:** por que o resultado ficou parecido com o da regressão logística, mesmo tendo duas camadas?

*(escreva aqui)*

## Tarefa 5 · Softmax na saída

A rede solta scores crus. Pra virar porcentagem, passa por `torch.softmax(scores, dim=1)`.

Escolha uma imagem do teste, mostre os 3 dígitos mais prováveis e veja se a rede acertou.

In [ ]:
i = 0
scores = minha_rede(X_te[i:i + 1]).detach()
# TODO: transforme em probabilidades e imprima os 3 maiores
# dica: probs.topk(3) devolve valores e índices

plt.imshow(X_te[i].reshape(8, 8), cmap="gray_r")
plt.axis("off")
plt.show()
print("certo:", y_te[i].item())

## Desafio 1 · Larga ou funda?

Com mais ou menos o mesmo número de parâmetros, o que rende mais: uma camada escondida grande ou três pequenas?

Use `n_parametros(modelo)` pra comparar de forma justa.

In [ ]:
# TODO: monte pelo menos dois modelos, imprima parâmetros e acurácia de cada um

## Desafio 2 · Inicialização

Zere todos os pesos com `nn.init.zeros_(camada.weight)` e treine de novo.

O que acontece, e por quê? (dica: se todos os neurônios da camada começam iguais, eles recebem a mesma correção)

⚠️ **Neste desafio, chame `treinar(zerada, epocas=100, lr=0.1, reiniciar=False)`.**
Sem o `reiniciar=False` a função sorteia pesos novos antes do primeiro passo e
joga fora os zeros que você acabou de pôr — a rede treina como se nada tivesse
acontecido e chega aos mesmos 0,97 de sempre, escondendo o efeito que o
exercício quer te mostrar.


In [ ]:
# TODO